In [ ]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt

#### Load the data

In [ ]:
from ucimlrepo import fetch_ucirepo

breast_cancer = fetch_ucirepo(id=17)

### Explore the Data

In [ ]:
X = breast_cancer.data.features
y = breast_cancer.data.targets.squeeze()
X.shape, y.shape, X.info(), y.info(), y.value_counts()

#### Check for duplicates

In [ ]:
X.duplicated().sum()

np.int64(0)

In [ ]:
y = y.replace({"B": 0, "M": 1})
y = y.astype(int)

#### Split the dataset

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((455, 30), (455,), (114, 30), (114,))

**Preprocessing**: The dataset contains no missing values, and all predictor variables are numeric, so imputation and one-hot encoding are not required. Standardization is applied within the pipelines for Logistic Regression and SVC. Tree-based models (Random Forest, Gradient Boosting, and XGBoost) are evaluated without feature scaling.

### Define models, apply transformation, and evaluate the models

**Define Candidate Model Pipelines**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb

GB_pipeline = Pipeline([
    ("clf", GradientBoostingClassifier(random_state=42))
])

XGB_pipeline = Pipeline([
    ("clf", xgb.XGBClassifier(random_state=42))
])

LR_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

RF_pipeline = Pipeline([
    ('clf', RandomForestClassifier(random_state=42))
])

SVC_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ('clf', SVC(kernel='rbf', C=1.0, gamma='scale'))
])

### Standardized Cross Validation Evaluation

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
import time
import numpy as np

skfolds = StratifiedKFold(n_splits=3, random_state=42, shuffle=True)


In [ ]:
def evaluate_model(model_name, model, x, y, cv):

    y_true_all = []
    y_pred_all = []
    y_score_all = []

    start_time = time.perf_counter()
    fit_count = 0

    for train_index, val_index in cv.split(X=x, y=y):

        clone_clf = clone(model)

        X_train_folds = x.iloc[train_index]
        y_train_folds = y.iloc[train_index]

        X_val_folds = x.iloc[val_index]
        y_val_folds = y.iloc[val_index]

        # ONE fit for this fold
        clone_clf.fit(X_train_folds, y_train_folds)
        fit_count += 1

        # Class predictions
        y_pred = clone_clf.predict(X_val_folds)

        # Continuous scores for AUC
        if hasattr(clone_clf, "predict_proba"):
            y_score = clone_clf.predict_proba(X_val_folds)[:, 1]

        elif hasattr(clone_clf, "decision_function"):
            y_score = clone_clf.decision_function(X_val_folds)

        else:
            raise ValueError(
                f"{model_name} cannot produce scores for AUC"
            )

        # Save this fold's results
        y_true_all.extend(y_val_folds)
        y_pred_all.extend(y_pred)
        y_score_all.extend(y_score)

    runtime = time.perf_counter() - start_time

    # Convert to arrays
    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_score_all = np.array(y_score_all)

    # Metrics
    accuracy = accuracy_score(y_true_all, y_pred_all)
    precision = precision_score(y_true_all, y_pred_all)
    recall = recall_score(y_true_all, y_pred_all)
    f1 = f1_score(y_true_all, y_pred_all)
    auc = roc_auc_score(y_true_all, y_score_all)

    cm = confusion_matrix(
        y_true=y_true_all,
        y_pred=y_pred_all
    )

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "AUC": auc,
        "Runtime_seconds": runtime,
        "Model_evaluations": 1,
        "CV_fits": fit_count,
        "Confusion_matrix": cm
    }

In [ ]:
results = []

results.append(
    evaluate_model( "LR", LR_pipeline, X_train, y_train, skfolds)
)

results.append(
    evaluate_model( "SVC", SVC_pipeline, X_train, y_train, skfolds)
)

results.append(
    evaluate_model( "RF", RF_pipeline, X_train, y_train, skfolds)
)

results.append(
    evaluate_model( "GB", GB_pipeline, X_train, y_train, skfolds)
)

results.append(
    evaluate_model( "XGB", XGB_pipeline, X_train, y_train, skfolds)
)

In [ ]:
results_df = pd.DataFrame(results)
results_df[
    [
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "AUC",
        "Runtime_seconds",
        "Model_evaluations",
        "CV_fits"
    ]
]

,Model,Accuracy,Precision,Recall,F1,AUC,Runtime_seconds,Model_evaluations,CV_fits
0,LR,0.973626,0.970238,0.958824,0.964497,0.995191,0.100289,1,3
1,SVC,0.962637,0.963636,0.935294,0.949254,0.993127,0.029050,1,3
2,RF,0.960440,0.947059,0.947059,0.947059,0.990041,0.507892,1,3
3,GB,0.949451,0.934911,0.929412,0.932153,0.984499,0.727525,1,3
4,XGB,0.956044,0.957317,0.923529,0.940120,0.989474,0.825145,1,3


: 